# AlphaFlow / ESMFlow on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bjing2016/alphaflow/blob/master/notebooks/AlphaFlow_Colab.ipynb)

**AlphaFlow** is a modified version of AlphaFold, fine-tuned with a flow matching objective for generative modeling of protein conformational ensembles. **ESMFlow** is the corresponding fine-tuned variant of ESMFold.

* Paper: [AlphaFold Meets Flow Matching for Generating Protein Ensembles](https://arxiv.org/abs/2402.04845) (Jing, Berger, Jaakkola, ICML 2024)
* Repository: [github.com/bjing2016/alphaflow](https://github.com/bjing2016/alphaflow)

This notebook walks through the complete workflow on a Google Colab GPU runtime:

1. **Setup** — verify GPU/toolchain, pip install AlphaFlow + OpenFold + dependencies (torch 2.4.1 / CUDA 12 — works on T4, A100, L4, and H100).
2. **Get the code & model weights** — clone the repository and download any of the 14 published checkpoints.
3. **Prepare inputs** — build a CSV of sequences and (for AlphaFlow) generate MSAs via the ColabFold MMseqs2 server. Optionally supply template PDBs for the MD+Templates models.
4. **Run inference** — sample conformational ensembles with the standard or distilled models, with knobs for `--samples`, `--steps`, `--tmax`, `--self_cond`, `--resample`, `--noisy_first`, `--no_diffusion`, etc.
5. **Visualize** — inspect the generated ensemble interactively with py3Dmol.
6. **Evaluate** — reproduce the ATLAS ensemble analysis pipeline (`analyze_ensembles` + `print_analysis`).
7. **Train / fine-tune** — reference the training commands for users with their own datasets.

> ⚠️ **Runtime requirements.** Pick **Runtime → Change runtime type → T4 / L4 / A100 / H100 GPU** before running. Cells in *Section 1* take ~5–10 minutes the first time because OpenFold is built from source. Blackwell-class GPUs (the new "G4" tier) are not supported by this notebook — see Section 1.3 for the reason.

---
## 1. Environment setup

Modern Colab runtimes already ship Python 3.10+/CUDA 12/gcc-11 — everything AlphaFlow needs once we bump its 2023 dependency pins. We skip condacolab entirely (recent condacolab versions silently install Python 3.12 even when asked for 3.9) and just `pip install` versions of `torch`, `numpy`, `pytorch_lightning`, `mdtraj`, etc. that match the Colab runtime.

### 1.1 Confirm GPU availability

In [ ]:
!nvidia-smi

### 1.2 Toolchain check

Confirm Colab's bundled Python, CUDA, and host compiler are intact before installing the Python deps. No installs happen here, no kernel restart.

In [ ]:
import sys
print('Python:', sys.version.split()[0])
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader
!nvcc --version 2>/dev/null | tail -n2 | head -n1 || echo 'nvcc missing — install Colab CUDA toolkit'
!gcc --version | head -n1

### 1.3 Install Python packages

Colab's runtime ships Python 3.12 with torch 2.5/2.6, numpy 2.0.2, scipy 1.16.x, pandas 2.2.x, dm-tree 0.1.10. The plan:

* **Force-reinstall torch as `2.3.1+cu121`** — last release with broadly available `+cu121` wheels that openfold@103d037 has been validated against.
* **Leave numpy / scipy / pandas / dm-tree alone.** Downgrading numpy from 2.0.2 → 1.26.4 (an earlier revision of this notebook) silently breaks torch's numpy ABI, which makes openfold's wheel build fail.
* **Clone openfold locally and patch it before building**:
  - Replace `compute_capabilities` in `setup.py` with a CUDA-12-safe set (drops `sm_37/52/61`, which `nvcc` from CUDA 12 rejects).
  - Rewrite `np.object → object` in `openfold/data/templates.py` (numpy 2.0 removed `np.object`; openfold issue #391). AlphaFlow imports this transitively at predict time.
* **Build with `--no-build-isolation`** — otherwise pip creates a sandbox env without torch and the wheel build fails before `nvcc` is even invoked, on `from torch.utils.cpp_extension import CUDAExtension`.
* **Pipe build output through `tail`** so any remaining error surfaces in the cell output instead of being hidden behind pip's "See above for output".

| | Colab default | This notebook |
|---|---|---|
| Python | 3.12 | unchanged |
| `torch` | `2.5.x / 2.6.x` (varies) | **`2.3.1+cu121`** (covers T4/A100/L4/H100) |
| `numpy` / `scipy` / `pandas` / `dm-tree` | 2.0.2 / 1.16.x / 2.2.x / 0.1.10 | **unchanged** |
| `pytorch_lightning` | not installed | **`2.3.3`** |
| `mdtraj` / `biopython` / `modelcif` / `ml-collections` / `fair-esm` | not installed | latest |
| `openfold` | not installed | **`@103d037`** + patched setup.py + patched templates.py |

Allow ~5–10 minutes for the OpenFold extension build.

> **Blackwell (G4):** still requires AlphaFlow [PR #58](https://github.com/bjing2016/alphaflow/pull/58) + torch 2.7+cu128 + openfold v2.2.0. Not implemented here.

In [ ]:
# 1. Pin torch back to 2.3.1+cu121 (openfold@103d037 has been validated against
#    this; Colab's pre-installed torch is replaced via --force-reinstall).
!python -m pip install --no-cache-dir --force-reinstall \
    torch==2.3.1+cu121 \
    -f https://download.pytorch.org/whl/torch_stable.html

# 2. AlphaFlow-specific Python deps that aren't already in Colab. We do NOT
#    downgrade numpy / scipy / pandas / dm-tree — Colab's 2.x defaults are
#    what torch 2.3.1's wheel ABI expects.
!python -m pip install --no-cache-dir --upgrade \
    biopython modelcif ml-collections absl-py einops fair-esm \
    setuptools wheel ninja
!python -m pip install --no-cache-dir mdtraj==1.10.2 pytorch_lightning==2.3.3

# 3. Make libcuda.so reachable so openfold's setup.py can read the GPU's
#    compute capability via cuDeviceGetAttribute. Without this it falls
#    through to a deprecated arch list (sm_37/52/61) that CUDA 12.x rejects
#    with "Unsupported gpu architecture compute_37".
!ln -sf /usr/lib/x86_64-linux-gnu/libcuda.so.1 \
        /usr/lib/x86_64-linux-gnu/libcuda.so 2>/dev/null || true

# 4. Clone openfold locally so we can patch it BEFORE the wheel build.
import os, re
OF_SRC = '/tmp/openfold_src'
!rm -rf {OF_SRC}
!git clone -q https://github.com/aqlaboratory/openfold.git {OF_SRC}
!cd {OF_SRC} && git checkout -q 103d037

# 4a. Defensive: rewrite setup.py's compute_capabilities to a CUDA-12-safe set
#     (drops sm_37/52/61). get_nvidia_cc() will still override this with the
#     runtime GPU's CC if libcuda.so resolves, but if it doesn't, this fallback
#     keeps the build from failing.
setup_path = f'{OF_SRC}/setup.py'
with open(setup_path) as f:
    src = f.read()
src = re.sub(
    r'compute_capabilities\s*=\s*\{[^}]*\}',
    'compute_capabilities = {(7, 0), (7, 5), (8, 0), (8, 6), (8, 9), (9, 0)}',
    src,
)
with open(setup_path, 'w') as f:
    f.write(src)

# 4b. Hard fix: openfold/data/templates.py uses np.object (removed in numpy 2.0;
#     openfold issue #391). AlphaFlow imports this at predict time, so patch
#     it BEFORE building so the installed package has the fix baked in.
!sed -i 's/np\.object\b/object/g' {OF_SRC}/openfold/data/templates.py
# Defensive: also patch the relax module (not on predict path but used by eval).
!sed -i 's/np\.bool\b/bool/g; s/np\.object\b/object/g' \
    {OF_SRC}/openfold/np/relax/utils.py 2>/dev/null || true

# 5. Build openfold. --no-build-isolation lets the build see the torch we just
#    installed; otherwise pip creates a sandbox env where torch isn't present
#    and `from torch.utils.cpp_extension import CUDAExtension` fails. Pipe
#    through `tail` so any remaining nvcc/gcc error actually surfaces in the
#    cell output instead of being hidden behind "See above for output".
os.environ.setdefault('CUDA_HOME', '/usr/local/cuda')
!CUDA_HOME=$CUDA_HOME python -m pip install --no-cache-dir --no-build-isolation \
    {OF_SRC} 2>&1 | tail -60

# 6. Insurance: stereo_chemical_props.txt for analyze_ensembles / train.py.
import openfold
of_dir = os.path.dirname(openfold.__file__)
res_dir = os.path.join(of_dir, 'resources')
os.makedirs(res_dir, exist_ok=True)
stereo_path = os.path.join(res_dir, 'stereo_chemical_props.txt')
if not os.path.exists(stereo_path):
    !wget -q -O {stereo_path} https://git.scicore.unibas.ch/schwede/openstructure/-/raw/7102c63615b64735c4941278d92b554ec94415f8/modules/mol/alg/src/stereo_chemical_props.txt

In [ ]:
# Light-weight extras used by the rest of the notebook
!python -m pip install --no-cache-dir py3Dmol requests tqdm matplotlib

### 1.4 Clone the AlphaFlow repository

If you opened this notebook from a working clone you can skip this cell.

In [ ]:
import os
REPO_DIR = '/content/alphaflow'
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 https://github.com/bjing2016/alphaflow.git {REPO_DIR}
%cd {REPO_DIR}
!ls

In [ ]:
# Smoke test: verify CUDA is visible, the AlphaFlow package imports, and the
# detected GPU's compute capability is in the set we built kernels for.
import torch, sys
print('Python :', sys.version.split()[0])
print('Torch  :', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability(0)
    print('Device :', torch.cuda.get_device_name(0), f'(sm_{cc[0]}{cc[1]})')
    supported = {(7,5), (8,0), (8,6), (8,9), (9,0)}
    if cc not in supported:
        print(f'!! sm_{cc[0]}{cc[1]} is outside the gencode list this build covers.')
        print('   Blackwell (sm_100/sm_120) needs the alternate path noted in cell 1.3.')
import alphaflow, openfold
print('AlphaFlow & OpenFold imported successfully')

---
## 2. Choose a model and download the weights

All checkpoints live on Hugging Face under [bjing-mit/alphaflow](https://huggingface.co/bjing-mit/alphaflow). Pick one of the variants below. The **distilled** variants are 5–10× faster but slightly less accurate. The **12l** MD+Templates models are 2.5× faster than the 48-layer ones with a small accuracy hit.

| Family | Variant | What it models |
|---|---|---|
| `alphaflow_pdb` | base / distilled | Experimental ensembles (X-ray / cryo-EM) |
| `alphaflow_md` | base / distilled | MD trajectories at 300 K |
| `alphaflow_md_templates` | base / distilled / 12l-base / 12l-distilled | MD ensembles conditioned on a reference PDB |
| `esmflow_pdb` | base / distilled | Same as AlphaFlow-PDB but no MSA needed |
| `esmflow_md` | base / distilled | Same as AlphaFlow-MD but no MSA needed |
| `esmflow_md_templates` | base / distilled | Same as AlphaFlow-MD+Templates but no MSA needed |

In [ ]:
# @title Select a model { display-mode: "form" }
MODEL = "alphaflow_pdb_base_202402"  # @param ["alphaflow_pdb_base_202402","alphaflow_pdb_distilled_202402","alphaflow_md_base_202402","alphaflow_md_distilled_202402","alphaflow_md_templates_base_202402","alphaflow_md_templates_distilled_202402","alphaflow_12l_md_templates_base_202406","alphaflow_12l_md_templates_distilled_202406","esmflow_pdb_base_202402","esmflow_pdb_distilled_202402","esmflow_md_base_202402","esmflow_md_distilled_202402","esmflow_md_templates_base_202402","esmflow_md_templates_distilled_202402"]

import os, urllib.request
os.makedirs('weights', exist_ok=True)
WEIGHTS_PATH = f'weights/{MODEL}.pt'
URL = f'https://huggingface.co/bjing-mit/alphaflow/resolve/main/params/{MODEL}.pt'

if not os.path.exists(WEIGHTS_PATH):
    print(f'Downloading {URL}\n  -> {WEIGHTS_PATH}')
    urllib.request.urlretrieve(URL, WEIGHTS_PATH)

# Convenience flags derived from the chosen checkpoint
MODE = 'esmfold' if MODEL.startswith('esmflow') else 'alphafold'
IS_DISTILLED = 'distilled' in MODEL
USES_TEMPLATES = 'templates' in MODEL
IS_PDB_MODEL = '_pdb_' in MODEL
print('Mode:', MODE, '| distilled:', IS_DISTILLED, '| templates:', USES_TEMPLATES, '| PDB model:', IS_PDB_MODEL)
print('Weights size:', round(os.path.getsize(WEIGHTS_PATH) / 1e9, 2), 'GB')

---
## 3. Prepare inputs

Inference reads a CSV with two required columns: `name` and `seqres`. You can either:

* edit the in-cell list below, **or**
* set `UPLOAD_CSV = True` to pop a Colab upload widget for your own CSV.

The CSV `name` column is what every downstream filename keys off (MSA, template, output PDB), so keep names filesystem-safe.

In [ ]:
# @title Define the proteins to fold { display-mode: "form" }
import pandas as pd, os, shutil
os.makedirs('user_inputs', exist_ok=True)
input_csv = 'user_inputs/sequences.csv'

UPLOAD_CSV = False  # @param {type:"boolean"}

if UPLOAD_CSV:
    from google.colab import files
    print('Upload a CSV with `name,seqres` columns:')
    uploaded = files.upload()
    src_name = next(iter(uploaded))
    shutil.move(src_name, input_csv)
else:
    sequences = [
        # Example: ATLAS test target 6uof_A (~119 residues)
        ('6uof_A', 'DILTVEKLYRPSHEYGFLRETDTVKDYLDLVRKNRSSRFPVINQHQVVVGVVTMRDAGDKSPSTTIDKVMSRSLFLVGLSTNIANVSQRMIAEDFEMVPVVRSNQTLLGVVTRRDVMEK'),
    ]
    pd.DataFrame(sequences, columns=['name', 'seqres']).to_csv(input_csv, index=False)

df = pd.read_csv(input_csv)
assert {'name', 'seqres'}.issubset(df.columns), "CSV must contain `name` and `seqres` columns"
print(f'Loaded {len(df)} sequence(s) from {input_csv}')
df

### 3.1 Generate MSAs (AlphaFlow only)

AlphaFlow needs a multiple sequence alignment per target. The repo bundles `scripts/mmseqs_query.py`, which queries the public ColabFold MMseqs2 server and writes alignments to `{alignment_dir}/{name}/a3m/{name}.a3m`.

**Skip this cell entirely if you selected an ESMFlow model** — ESMFlow folds directly from sequence.

In [ ]:
MSA_DIR = 'user_inputs/msa_dir'
if MODE == 'alphafold':
    !python -m scripts.mmseqs_query --split {input_csv} --outdir {MSA_DIR}
    !ls {MSA_DIR}
else:
    print('Skipping MSA generation (ESMFlow model selected).')

### 3.2 (Optional) Provide template structures

For the `*_md_templates_*` checkpoints you can condition each prediction on a reference structure (e.g. an experimental PDB or an AlphaFold prediction). The model expects:

* one structure per CSV row, named `{name}.pdb`,
* a single chain with no residue gaps,
* sequence matching the CSV's `seqres`.

The cell below supports three input modes via the `TEMPLATE_SOURCE` switch:

* `"upload"` — drop your own files via Colab's upload dialog. Both **`.pdb`** and **`.cif`/`.mmcif`** are accepted; mmCIF is converted to single-chain PDB on the fly with `mdtraj`.
* `"rcsb"` — pull straight from the RCSB by `{pdbid}_{chain}` name (the default; useful for the example sequence).
* `"none"` — skip the section entirely (also auto-skipped for non-templates checkpoints).

In [ ]:
# @title Template structures { display-mode: "form" }
TEMPLATE_SOURCE = "rcsb"  # @param ["upload", "rcsb", "none"]

import os, urllib.request, shutil
TEMPLATES_DIR = 'user_inputs/templates'

def _trim_to_chain(in_path, out_path, chain_id):
    """Keep only ATOM/TER/END lines for `chain_id` (col 22) — single-chain, no HETATM."""
    with open(in_path) as f, open(out_path, 'w') as g:
        for line in f:
            rec = line[:6].strip()
            if rec in ('ATOM', 'TER') and (len(line) < 22 or line[21] == chain_id):
                g.write(line)
            elif rec == 'END':
                g.write(line)

def _cif_to_single_chain_pdb(cif_path, pdb_path, chain_id=None):
    """Convert mmCIF -> single-chain PDB via mdtraj."""
    import mdtraj as md
    traj = md.load(cif_path)
    if chain_id is not None:
        # mdtraj uses integer chain indices; map A->0, B->1, ... as a heuristic
        idx = ord(chain_id.upper()) - ord('A')
        sel = traj.topology.select(f'chainid {idx} and protein')
    else:
        sel = traj.topology.select('chainid 0 and protein')
    traj.atom_slice(sel).save_pdb(pdb_path)

if not USES_TEMPLATES or TEMPLATE_SOURCE == 'none':
    TEMPLATES_DIR = None
    print('Skipping templates (model does not use them, or TEMPLATE_SOURCE="none").')
elif TEMPLATE_SOURCE == 'upload':
    os.makedirs(TEMPLATES_DIR, exist_ok=True)
    from google.colab import files
    print('Upload one structure per CSV row. Filename stem must match the CSV `name`.')
    print('Accepted extensions: .pdb, .cif, .mmcif (mmCIF will be converted to single-chain PDB).')
    uploaded = files.upload()
    for fname in uploaded:
        stem, ext = os.path.splitext(fname)
        ext = ext.lower()
        # Allow `<name>_<chain>.pdb` form: user can encode chain via underscore (matches RCSB convention)
        chain_id = stem.split('_')[-1] if '_' in stem and len(stem.split('_')[-1]) == 1 else None
        out_pdb = f'{TEMPLATES_DIR}/{stem}.pdb'
        if ext == '.pdb':
            if chain_id:
                _trim_to_chain(fname, out_pdb, chain_id)
            else:
                shutil.move(fname, out_pdb)
        elif ext in ('.cif', '.mmcif'):
            _cif_to_single_chain_pdb(fname, out_pdb, chain_id)
            os.remove(fname)
        else:
            print(f'Skipping {fname}: unsupported extension')
            continue
        print(f'  -> {out_pdb}')
elif TEMPLATE_SOURCE == 'rcsb':
    os.makedirs(TEMPLATES_DIR, exist_ok=True)
    # For each CSV row whose `name` looks like `{pdbid}_{chain}`, fetch from RCSB
    for row_name in pd.read_csv(input_csv)['name']:
        out_pdb = f'{TEMPLATES_DIR}/{row_name}.pdb'
        if os.path.exists(out_pdb):
            continue
        if '_' not in row_name:
            print(f'  ! {row_name}: no chain id encoded in name, skipping RCSB fetch')
            continue
        pdb_id, chain_id = row_name.rsplit('_', 1)
        full = out_pdb + '.full'
        urllib.request.urlretrieve(f'https://files.rcsb.org/download/{pdb_id}.pdb', full)
        _trim_to_chain(full, out_pdb, chain_id)
        os.remove(full)
        print(f'  fetched {pdb_id} chain {chain_id} -> {out_pdb}')

if TEMPLATES_DIR:
    print('\nFiles in', TEMPLATES_DIR + ':')
    for f in sorted(os.listdir(TEMPLATES_DIR)):
        print(' ', f)

---
## 4. Run inference

The full CLI lives in `predict.py`. Important flags:

* `--samples N` — number of conformations to draw per target (each becomes a model in the output PDB).
* `--steps S` — number of flow-matching integration steps (defaults to 10).
* `--tmax T` — truncate the integration at `t = T` to trade diversity for accuracy. Together `--tmax 0.2 --steps 2` is a good high-precision setting.
* `--self_cond` + `--resample` — recommended for **PDB** models for slightly better quality.
* `--noisy_first` + `--no_diffusion` — **required** for any **distilled** checkpoint.
* `--templates_dir DIR` — only for the `*_md_templates_*` checkpoints.
* `--pdb_id NAME [NAME ...]` — restrict inference to a subset of CSV rows.
* `--subsample N` — randomly subsample the MSA to N sequences (see paper Appendix B.1).
* `--runtime_json PATH` — dump per-target wallclock timings.

In [ ]:
# @title Sampling parameters { display-mode: "form" }
SAMPLES = 5    # @param {type:"integer"}
STEPS = 10     # @param {type:"integer"}
TMAX = 1.0     # @param {type:"number"}
OUTPDB = f'outpdb/{MODEL}'
os.makedirs(OUTPDB, exist_ok=True)

cmd = [
    'python', 'predict.py',
    '--mode', MODE,
    '--input_csv', input_csv,
    '--weights', WEIGHTS_PATH,
    '--samples', str(SAMPLES),
    '--steps', str(STEPS),
    '--tmax', str(TMAX),
    '--outpdb', OUTPDB,
]
if MODE == 'alphafold':
    cmd += ['--msa_dir', MSA_DIR]
if USES_TEMPLATES and TEMPLATES_DIR:
    cmd += ['--templates_dir', TEMPLATES_DIR]
if IS_PDB_MODEL and not IS_DISTILLED:
    cmd += ['--self_cond', '--resample']  # recommended PDB-model quality boost
if IS_DISTILLED:
    cmd += ['--noisy_first', '--no_diffusion']  # required for distilled checkpoints

print(' '.join(cmd))
import subprocess
subprocess.run(cmd, check=True)
!ls {OUTPDB}

Each output PDB contains `--samples` models concatenated with `MODEL` records, ready to load into PyMOL/ChimeraX/VMD or to stream as a trajectory.

---
## 5. Visualize the ensemble

We use [py3Dmol](https://pypi.org/project/py3Dmol/) to overlay the sampled conformations directly in the notebook. Use the slider in the viewer's controls to switch models, or render all of them simultaneously by setting `OVERLAY = True`.

In [ ]:
import os, glob, py3Dmol

TARGET = pd.read_csv(input_csv).iloc[0]['name']  # first sequence by default
pdb_path = f'{OUTPDB}/{TARGET}.pdb'
with open(pdb_path) as f:
    pdb_str = f.read()

OVERLAY = True  # set False to scrub through models one at a time
view = py3Dmol.view(width=700, height=500)
view.addModelsAsFrames(pdb_str, 'pdb')
view.setStyle({'model': -1}, {'cartoon': {'color': 'spectrum'}})
if OVERLAY:
    view.animate({'loop': 'forward', 'interval': 200})
view.zoomTo()
view.show()

In [ ]:
# Quick numerical look: per-residue C-alpha RMSF across the ensemble
import mdtraj as md
import numpy as np, matplotlib.pyplot as plt

traj = md.load(pdb_path)
ca = traj.atom_slice(traj.topology.select('name CA'))
ca.superpose(ca, 0)
rmsf = md.rmsf(ca, ca, 0) * 10  # nm → Å

plt.figure(figsize=(8, 3))
plt.plot(np.arange(1, len(rmsf) + 1), rmsf)
plt.xlabel('Residue'); plt.ylabel('Cα RMSF (Å)')
plt.title(f'{TARGET} ensemble flexibility ({traj.n_frames} samples)')
plt.tight_layout(); plt.show()

In [ ]:
# Download all generated PDBs as a single zip
import shutil
zip_path = f'/content/{MODEL}_ensembles.zip'
shutil.make_archive(zip_path[:-4], 'zip', OUTPDB)
from google.colab import files
files.download(zip_path)

---
## 6. (Optional) Reproduce ATLAS evaluations

The repo's `scripts/analyze_ensembles.py` compares a directory of generated ensembles against ATLAS reference MD trajectories. The full ATLAS dataset is ~600 GB and well beyond Colab disk; this section is mostly here as a reference. Run it locally on a workstation or, on Colab, point `--atlas_dir` at a single downloaded target.

```bash
# 1. Download (a subset of) ATLAS targets
bash scripts/download_atlas.sh

# 2. Sample 250 conformations per target with the chosen model (see Section 4)
python predict.py --mode alphafold --input_csv splits/atlas_test.csv \
    --msa_dir [MSA_DIR] --weights weights/alphaflow_md_base_202402.pt \
    --samples 250 --outpdb outpdb/atlas_test

# 3. Compute per-target metrics
python -m scripts.analyze_ensembles \
    --atlas_dir [ATLAS_DIR] --pdbdir outpdb/atlas_test --num_workers 4

# 4. Print a comparison across models
python -m scripts.print_analysis outpdb/atlas_test/out.pkl [other_out.pkl ...]
```

If you only need pairwise ensemble metrics (no MD reference), `analyze_ensembles.py` still produces the AF-side statistics (`af_pairwise`, `af_rmsf`, ...).

---
## 7. (Optional) Training and fine-tuning

Training requires the full PDB mmCIF dump + OpenProteinSet MSAs (or the ATLAS preprocessed NPZs). These do not fit on Colab; the cell below is a reference of the canonical commands so the notebook still covers the full functionality of the codebase.

```bash
# Download AlphaFold params used to initialize training
wget https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar
tar -xvf alphafold_params_2022-12-06.tar params_model_1.npz
wget https://dl.fbaipublicfiles.com/fair-esm/models/esmfold_3B_v1.pt

# AlphaFlow-PDB base run
python train.py --lr 5e-4 --noise_prob 0.8 --accumulate_grad 8 \
    --train_epoch_len 80000 --train_cutoff 2018-05-01 --filter_chains \
    --train_data_dir [PDB_NPZ_DIR] --train_msa_dir [OPENFOLD_DIR] \
    --mmcif_dir [MMCIF_DIR] --val_msa_dir [VAL_MSA_DIR] \
    --run_name alphaflow_pdb_base [--wandb]

# Continue on ATLAS for the MD model
python train.py --normal_validate --sample_train_confs --sample_val_confs \
    --num_val_confs 100 --pdb_chains splits/atlas_train.csv --val_csv splits/atlas_val.csv \
    --self_cond_prob 0.0 --noise_prob 0.9 --val_freq 10 --ckpt_freq 10 \
    --train_data_dir [ATLAS_NPZ_DIR] --train_msa_dir [ATLAS_MSA_DIR] \
    --ckpt weights/alphaflow_pdb_base_202402.pt --run_name alphaflow_md

# Add templates: append --first_as_template --extra_input --lr 1e-4 \
#                       --restore_weights_only --extra_input_prob 1.0
# Distill any model: append --distillation --ckpt [PATH]
# ESMFlow training:  use --mode esmfold --train_cutoff 2020-05-01
```

See the [README](https://github.com/bjing2016/alphaflow#training) for dataset preparation steps (`scripts/unpack_mmcif.py`, `scripts/add_msa_info.py`, `scripts/cluster_chains.py`, `scripts/prep_atlas.py`).

---
## 8. Pre-computed ensembles

If you only want to inspect the ensembles published with the paper, every model's 250-sample ATLAS / PDB prediction set is downloadable directly:

```python
ENSEMBLE = 'alphaflow_md_base_202402'  # or any name from the README's 'Ensembles' table
!wget https://huggingface.co/bjing-mit/alphaflow/resolve/main/samples/{ENSEMBLE}.zip
!unzip -q {ENSEMBLE}.zip -d {ENSEMBLE}
```

These can then be plugged into Sections 5 and 6 above without rerunning inference.

---
## Citation

```bibtex
@inproceedings{jing2024alphafold,
  title={AlphaFold Meets Flow Matching for Generating Protein Ensembles},
  author={Jing, Bowen and Berger, Bonnie and Jaakkola, Tommi},
  year={2024},
  booktitle={Forty-first International Conference on Machine Learning}
}
```